# Data Preprocessing - Gallstone Dataset This notebook handles data loading, exploration, split, scaling, and feature bounds computation for the Gallstone dataset.


## 1. Import Libraries


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import pickle
import os
import warnings
warnings.filterwarnings('ignore')


## 2. Load Dataset


In [2]:
# Load the dataset
df = pd.read_csv('./data/dataset-uci.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()[:10]}... (total {len(df.columns)})")
print(f"\nFirst few rows:")
df.head()


Dataset shape: (319, 39)

Columns: ['Gallstone Status', 'Age', 'Gender', 'Comorbidity', 'Coronary Artery Disease (CAD)', 'Hypothyroidism', 'Hyperlipidemia', 'Diabetes Mellitus (DM)', 'Height', 'Weight']... (total 39)

First few rows:


,Gallstone Status,Age,Gender,Comorbidity,Coronary Artery Disease (CAD),Hypothyroidism,Hyperlipidemia,Diabetes Mellitus (DM),Height,Weight,...,High Density Lipoprotein (HDL),Triglyceride,Aspartat Aminotransferaz (AST),Alanin Aminotransferaz (ALT),Alkaline Phosphatase (ALP),Creatinine,Glomerular Filtration Rate (GFR),C-Reactive Protein (CRP),Hemoglobin (HGB),Vitamin D
0,0,50,0,0,0,0,0,0,185,92.8,...,40.0,134.0,20.0,22.0,87.0,0.82,112.47,0.0,16.0,33.0
1,0,47,0,1,0,0,0,0,176,94.5,...,43.0,103.0,14.0,13.0,46.0,0.87,107.10,0.0,14.4,25.0
2,0,61,0,0,0,0,0,0,171,91.1,...,43.0,69.0,18.0,14.0,66.0,1.25,65.51,0.0,16.2,30.2
3,0,41,0,0,0,0,0,0,168,67.7,...,59.0,53.0,20.0,12.0,34.0,1.02,94.10,0.0,15.4,35.4
4,0,42,0,0,0,0,0,0,178,89.6,...,30.0,326.0,27.0,54.0,71.0,0.82,112.47,0.0,16.8,40.6


## 3. Data Exploration & Class Balance


In [3]:
# Check target distribution
target_col = 'Gallstone Status'
print("Target distribution (Gallstone Status):")
print(df[target_col].value_counts())
print(f"\nPercentage:")
print(df[target_col].value_counts(normalize=True) * 100)

# Check missing values
print(f"Missing values count: {df.isnull().sum().sum()}")


Target distribution (Gallstone Status):
Gallstone Status
0    161
1    158
Name: count, dtype: int64

Percentage:
Gallstone Status
0    50.470219
1    49.529781
Name: proportion, dtype: float64
Missing values count: 0


## 4. Feature Selection & Split


In [4]:
# Separate features and target
X = df.drop(columns=[target_col])
y = df[target_col]

# Split into train-test sets (80% train, 20% test, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")


Training set: (255, 38)
Test set: (64, 38)


## 5. Feature Scaling


In [5]:
# Scale features using MinMaxScaler with range [-1, 1]
scaler = MinMaxScaler(feature_range=(-1, 1))
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled successfully.")


Features scaled successfully.


## 6. Compute Feature Bounds for DP


In [6]:
# Theoretical bounds from MinMaxScaler: every feature is in [-1, 1]
n_features = X_train_scaled.shape[1]
lower_bound = np.full(n_features, -1.0)
upper_bound = np.full(n_features, 1.0)
bounds = (lower_bound, upper_bound)
print(f"Theoretical feature bounds (data-independent):")
print(f"  Lower: [-1.0] × {n_features} features")
print(f"  Upper: [+1.0] × {n_features} features")


Theoretical feature bounds (data-independent):
  Lower: [-1.0] × 38 features
  Upper: [+1.0] × 38 features


## 7. Save Preprocessed Data


In [7]:
os.makedirs('data', exist_ok=True)

processed_data = {
    'X_train': X_train_scaled,
    'X_test': X_test_scaled,
    'y_train': y_train.values,
    'y_test': y_test.values,
    'feature_names': X.columns.tolist(),
    'bounds': bounds,
    'scaler': scaler
}

with open('data/processed_data.pkl', 'wb') as f:
    pickle.dump(processed_data, f)

print("✓ Preprocessed data saved to 'data/processed_data.pkl'")


✓ Preprocessed data saved to 'data/processed_data.pkl'
